# Big Data Management
## Univeristy of Toronto / University of Waterloo
### Winter 2026
### Instructor: Melissa Singh


## Team Members - Group 10
- Abduljalil Najar
- Haya Tariq
- Oleksiy Kovtonyuk
- Rosy Zhou
- Miguel Morales Gonzalez


---
## Project: Batch vs. Streaming Analytics at Scale: A Comparative Study Using Apache Spark

## Project Details:

**Project type:** Data analysis + scalable systems comparison  
**Topic:** *Batch vs. Streaming Analytics at Scale: A Comparative Study Using Apache Spark*  
**Dataset:** City of Chicago — *Traffic Crashes – Crashes* (Socrata ID: **85ca-t3if**)*  
**Scope:** **2022–2025** (inclusive)  
**Execution model:** **Hybrid** (4 Jupyter notebooks + 3 Python scripts)  
**Streaming method:** **File-based Spark Structured Streaming** (simulated incremental file arrival)  
**Primary use case:** **Time-based aggregations** (hourly event-time windows; plus daily/monthly batch rollups)  
**Comparison type:** **Behavioral comparison** (not strict output parity)  
**Evaluation dimensions:** Latency, Complexity, Scalability considerations, Fault tolerance

* Note: Please download the dataset CSV from the portal and place it in `data/raw/` as `Traffic_Crashes_-_Crashes.csv` before running the notebooks.
The portal URL is: https://data.cityofchicago.org/Transportation/Traffic-Crashes-Crashes/85ca-t3if

---

## Summary

This project implements and compares **batch** and **streaming** analytics using Apache Spark on a real-world public dataset. Both approaches consume the same canonical dataset (Silver Parquet, 2022–2025) to highlight differences in **system behavior** rather than forcing exact output parity.

- **Batch path (historical):** Spark DataFrames / Spark SQL compute complete aggregates over the full dataset.
- **Streaming path (incremental):** Spark Structured Streaming consumes files as they arrive and produces hourly event-time window aggregates.
- **Architecture framing:** The work follows a simple **Bronze → Silver → Gold** pattern and maps cleanly to a **Lambda Architecture** narrative (batch for completeness, streaming for time-to-first-result).

---

## 1. Repository Structure

> This reflects the final structure used for execution (as implemented).

```text
BIG-DATA_GROUP-10/
├── data/
│   ├── checkpoints
│   │   └──streaming_job/                  # Structured Streaming checkpoint state
│   ├── raw/
│   │   └── Traffic_Crashes_-_Crashes.csv  # Bronze CSV (unchanged)
│   ├── replay/                            # Generated daily replay chunks (Parquet)
│   ├── silver/                            # Canonical Silver Parquet
│   └── stream_in/                         # Streaming input directory (watched)
│
├── notebook/
│   ├── Big_Data-Management_Group10 - CoverPage.ipynb
│   ├── 01_bronze_to_silver.ipynb
│   ├── 02_batch_analysis.ipynb
│   ├── 03_streaming_logic_preview.ipynb
│   ├── 04_orchestration.ipynb
│
├── outputs/
│   ├── batch/                             # Batch Gold outputs
│   └── streaming/                         # Streaming Gold outputs
│
├── src/
│   ├── make_replay_files.py               # Silver → replay chunks
│   ├── dropper.py                         # replay → stream_in (simulated arrival)
│   └── streaming_job.py                   # Structured Streaming job
│
├── README.md
└── .gitignore
```

---

## 2. Responsibilities (Notebooks vs. Scripts)

### 2.1 Notebooks (4)

#### `01_bronze_to_silver.ipynb`
**Purpose:** Create canonical **Silver** dataset.

- Read raw CSV (Bronze)
- Select canonical schema (minimal columns)
- Parse `CRASH_DATE` to timestamp
- Filter to 2022–2025 and drop invalid timestamps
- Derive partition columns `year`, `month`
- Write partitioned Parquet to `data/silver/`

**Constraints:**
- No analytics/aggregations
- No streaming logic

---

#### `02_batch_analysis.ipynb` (Batch analytics baseline)
**Purpose:** Produce complete batch **Gold** outputs from Silver.

- Read `data/silver/` only
- Compute time-based aggregations (including hourly event-time windows)
- Write Gold outputs to `outputs/batch/`
- Print a clear batch run summary (rows, time range, elapsed time, outputs)

---

#### `03_streaming_logic_preview.ipynb` (Validation, not orchestration)
**Purpose:** Explain and validate streaming semantics.

- Does **not** run long-lived streaming queries
- Reads **already-produced** streaming outputs as batch (`spark.read.parquet("outputs/streaming")`)
- Validates window alignment (hourly boundaries) and basic sanity invariants
- Confirms streaming output schema matches the batch hourly window output structurally

---

#### `04_orchestration.ipynb` (Experiment logbook + comparison)
**Purpose:** Document the final experiment run and produce report-ready comparison notes.

- Records batch summary (from Notebook 02)
- Records streaming summary (from script execution + output inspection)
- Includes restart / checkpoint observations
- Contains the comparison table and key takeaways for the final report

**Constraint:** No `readStream`, no `awaitTermination`, no orchestration of long-running scripts from the notebook.

---

### 2.2 Scripts (3)

#### `src/make_replay_files.py`
**Purpose:** Generate replay chunks for simulated streaming.

- Reads Silver Parquet (`--input data/silver/...`)
- Creates deterministic chunks by **day** (or week)
- Writes to `data/replay/`

---

#### `src/dropper.py`
**Purpose:** Simulate data-in-motion.

- Copies one Parquet file every N seconds from `data/replay/` to `data/stream_in/`
- Uses atomic rename to ensure Spark reads complete files
- Ensures destination filenames are unique (prefixes with `chunk_key=YYYY-MM-DD__...`)

---

#### `src/streaming_job.py`
**Purpose:** Long-running Spark Structured Streaming consumer.

- Watches `data/stream_in/` for new Parquet files
- Uses event-time windows on `CRASH_DATE` (1 hour)
- Uses watermarking (1 hour)
- Writes Parquet outputs to `outputs/streaming/` (append mode)
- Uses checkpoints under `data/checkpoints/streaming_job/`

---

## 3. Data Model (Silver Schema)

### 3.1 Bronze
- Raw CSV downloaded from the dataset portal
- Stored unchanged in `data/raw/`

### 3.2 Silver (Canonical)
**Goal:** Keep only the minimum viable columns needed for time-based and context analysis.

**Columns retained:**
- `CRASH_RECORD_ID` (traceability only)
- `CRASH_DATE`
- `INJURIES_TOTAL`, `INJURIES_FATAL`, `MOST_SEVERE_INJURY`
- `WEATHER_CONDITION`, `LIGHTING_CONDITION`, `ROADWAY_SURFACE_COND`
- `PRIM_CONTRIBUTORY_CAUSE`, `SEC_CONTRIBUTORY_CAUSE`

**Storage:**
- Parquet, partitioned by `year` and `month`
- Location: `data/silver/`

### 3.3 Gold (Outputs)
- Batch Gold outputs: `outputs/batch/`
- Streaming Gold outputs: `outputs/streaming/`

---

## 4. Gold Outputs (Final)

### 4.1 Batch Gold outputs (written to `outputs/batch/`)
Final set produced by `02_batch_analysis.ipynb`:

- `crashes_by_hour_window`
- `crashes_by_date`
- `crashes_by_month`
- `injuries_by_month`
- `crashes_by_weather`
- `crashes_by_lighting`
- `crashes_by_roadway`

### 4.2 Streaming Gold outputs (written to `outputs/streaming/`)
Final set produced by `streaming_job.py`:

- `crashes_by_hour_window` (hourly event-time window aggregates)

> Note: We intentionally kept streaming scope focused on hourly window output to emphasize behavioral differences (time-to-first-result, incremental updates, checkpoint recovery).

---

## 5. Execution Flow

### 5.1 One-time preparation
1. Run `01_bronze_to_silver.ipynb` (creates Silver)
2. Run `02_batch_analysis.ipynb` (creates batch Gold outputs)

### 5.2 Streaming experiment setup
3. Generate replay chunks:
   ```bash
   python src/make_replay_files.py --input data/silver/crashes_parquet --output data/replay --chunking day --overwrite --silver data/silver/crashes_parquet
   ```

### 5.3 Streaming experiment (clean run)
4. Clear streaming state (optional but recommended for a clean comparison):
   - `outputs/streaming/`
   - `data/stream_in/`
   - `data/checkpoints/streaming_job/`

5. Start streaming consumer (Terminal A):
   ```bash
   python src/streaming_job.py --input data/stream_in --output outputs/streaming --checkpoint data/checkpoints/streaming_job --window-size "1 hour" --trigger "10 seconds"
   ```

6. Start dropper (Terminal B):
   ```bash
   python src/dropper.py --source data/replay --dest data/stream_in --interval-seconds 5
   ```

7. Let the system run for a short controlled period (e.g., a few minutes), then stop dropper.

8. (Fault tolerance demo) Stop and restart `streaming_job.py` using the **same checkpoint** path.

9. Run `03_streaming_logic_preview.ipynb` (validation) and `04_orchestration.ipynb` (summary + comparison).

---

## 6. Behavioral Comparison Framework (What We Measure)

- **Latency:** batch end-to-end runtime vs streaming time-to-first-output and incremental updates
- **Complexity:** number of components and operational steps
- **Scalability considerations:** bounded scan vs continuous ingestion; state size for windows
- **Fault tolerance:** batch rerun vs streaming checkpoint restart

---

## 7. Report Mapping (Word Doc)

The final report follows the required data analysis outline:

- **Objectives:** goals, questions, hypothesis, approach
- **Data Preparation:** data source, cleaning, schema selection, ETL challenges
- **Analysis:** batch outputs + streaming outputs + comparison across the four dimensions
- **Conclusions:** findings, trade-offs, limitations, and recommendations
- **References (MLA):** dataset URL + Spark docs + any external sources

---

## 8. References

- City of Chicago Open Data Portal — Traffic Crashes – Crashes (85ca-t3if):
  https://data.cityofchicago.org/Transportation/Traffic-Crashes-Crashes/85ca-t3if
- Apache Spark Structured Streaming Guide:
  https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html


---

## Project Notebooks - Additional Details

Please see below a description of each notebook. Once that the environment is set up, you can run the notebooks in the order listed below. Each notebook builds on the previous one. (Note: we have included the environment setup in the section below).

| Notebook | Description |
|---|---|
| `01_bronze_to_silver.ipynb` | **ETL layer.** Reads the raw Chicago Traffic Crashes CSV, cleans and standardises the schema, and writes canonical Silver Parquet partitioned by `year` and `month`. Run this once before anything else. |
| 02_batch_analysis.ipynb | **Batch layer.** Reads Silver Parquet and computes 5 core aggregations (hourly windows, daily counts, monthly trends, injury totals, contextual breakdowns). Writes 7 Gold outputs to batch and prints a full batch run summary. |
| 03_streaming_logic_preview.ipynb | **Streaming validation.** Reads the Parquet files produced by streaming_job.py as a static batch and validates window alignment, schema correctness, and data sanity. Run after the streaming scripts have produced output. |
| 04_orchestration.ipynb | **Experiment log.** Documents the batch and streaming experiments, captures metrics, and provides a structured batch vs. streaming comparison for the report. |

> **Streaming scripts** (run outside the notebooks, in two separate terminals):
> - make_replay_files.py: splits Silver into daily chunks under replay (run once)
> - dropper.py: feeds chunks into stream_in every N seconds
> - streaming_job.py: Spark Structured Streaming job; reads stream_in, aggregates hourly windows, writes to streaming

---

## Windows Environment Setup Guide

This section documents every step required to run this project on a **Windows machine** using a local Spark installation.


### Step 1 — Install Java 11 (JDK)

Spark requires Java 11. We used **Eclipse Temurin 11** (formerly AdoptOpenJDK).

**Option A — Install via winget (recommended):**
```powershell
winget install --id EclipseAdoptium.Temurin.11.JDK -e
```

**Option B — Manual install:**
Download from https://adoptium.net and choose **JDK 11, Windows x64**.

After installation, set the user environment variable:
```powershell
$jdk = Get-ChildItem "C:\Program Files\Eclipse Adoptium" -Directory |
  Where-Object { $_.Name -like "jdk-11*" } |
  Sort-Object Name -Descending |
  Select-Object -First 1

[Environment]::SetEnvironmentVariable("JAVA_HOME", $jdk.FullName, "User")

$path = [Environment]::GetEnvironmentVariable("Path", "User")
if ($path -notlike "*%JAVA_HOME%\bin*") {
  [Environment]::SetEnvironmentVariable("Path", "$path;%JAVA_HOME%\bin", "User")
}
```

Verify:
```powershell
java -version
# Expected: openjdk version "11.x.x" ...
```

---

### Step 2 — Install Hadoop Binaries (winutils)

Spark on Windows requires `winutils.exe` and `hadoop.dll` for local filesystem access. Without them, Spark throws native I/O errors when reading/writing Parquet.

1. Download the binaries matching your Spark's bundled Hadoop version.
   For **Spark 3.4.x** → use **Hadoop 3.3.x**.
   Trusted source: https://github.com/cdarlint/winutils

2. Place both files in `C:\hadoop\bin\`:
   ```
   C:\hadoop\bin\winutils.exe
   C:\hadoop\bin\hadoop.dll
   ```

3. Set environment variables (one-time, in PowerShell):
   ```powershell
   [Environment]::SetEnvironmentVariable("HADOOP_HOME", "C:\hadoop", "User")
   $path = [Environment]::GetEnvironmentVariable("Path", "User")
   [Environment]::SetEnvironmentVariable("Path", "$path;C:\hadoop\bin", "User")
   ```

> **Note:** Every notebook and script in this project also sets `HADOOP_HOME` programmatically via `os.environ` before Spark starts, so the pipeline works even if the system variable is not set persistently.

---

### Step 3 — Install Anaconda / Python Environment

Install **Anaconda** (https://www.anaconda.com/download) or use an existing Python 3.9+ environment.

Create and activate a dedicated environment (optional but recommended):
```powershell
conda create -n bigdata python=3.10
conda activate bigdata
```

---

### Step 4 — Install Python Dependencies

```powershell
pip install pyspark==3.4.0 findspark jupyterlab pandas matplotlib
```

| Package | Purpose |
|---|---|
| `pyspark==3.4.0` | Apache Spark Python API |
| `findspark` | Locates PySpark automatically in Jupyter |
| `jupyterlab` | Notebook environment |
| `pandas` | Used for `.toPandas()` calls in visualisation cells |
| `matplotlib` | Charts in the batch analysis notebook |

---

### Step 5 — Verify the Environment

Run this quick check in a Python script or notebook cell:

```python
import os
import findspark

os.environ["JAVA_HOME"]   = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"]        = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("test").getOrCreate()
spark.range(5).show()
spark.stop()
print("Environment OK")
```

Expected output:
```
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+
Environment OK
```

---

### Step 6 — Clone / Open the Project

```powershell
git clone https://github.com/miguel-tm/big-data_group-10.git
cd big-data_group-10
jupyter lab
```

Then run the notebooks **in order**:

| Order | Notebook / Script | Run once? | Description |
|---|---|---|---|
| 1 | `01_bronze_to_silver.ipynb` | Yes | ETL: raw CSV → canonical Silver Parquet |
| 2 | `02_batch_analysis.ipynb` | Yes | Batch Gold outputs + run summary |
| 3 | `src/make_replay_files.py` | Yes | Generates streaming replay chunks under `data/replay/` |
| 4 | `src/streaming_job.py` + `src/dropper.py` | Run together in two terminals | Live streaming simulation |
| 5 | `03_streaming_logic_preview.ipynb` | After streaming run | Validates streaming output correctness |
| 6 | `04_orchestration.ipynb` | Final | Experiment log and batch vs streaming comparison |

---

### Common Issues & Fixes

| Error | Cause | Fix |
|---|---|---|
| `winutils.exe not found` | `HADOOP_HOME` not set or wrong path | Verify `C:\hadoop\bin\winutils.exe` exists; check `os.environ["HADOOP_HOME"]` in the script |
| `JAVA_HOME not set` | Java not installed or env var missing | Complete Step 1; restart terminal and kernel |
| `Unable to infer schema for Parquet` | Reading empty or metadata-only directory | Ensure the upstream step ran and produced `.parquet` data files |
| `AnalysisException: path not found` | Relative path resolved from wrong working directory | Use absolute paths or the `PROJECT_ROOT` constant defined in each notebook |
| `.tmp file not found` during streaming | Dropper's in-flight temp file picked up by Spark scanner | `streaming_job.py` uses `.option("pathGlobFilter", "*.parquet")` to ignore `.tmp` files |

> **One-time pip install (if not using Anaconda):**
> ```powershell
> pip install pyspark==3.4.0 findspark jupyterlab pandas matplotlib
> ```

> **Why Hadoop/winutils?**
> Spark on Windows requires `winutils.exe` and `hadoop.dll` for local filesystem access (reading/writing Parquet, HDFS-style directory operations). Without them you will see `UnsatisfiedLinkError` or `NativeIO$Windows` errors. This is a Windows-only compatibility shim,  no actual Hadoop cluster is needed.